# Export Stage 2 (EfficientNet-B0 + VBLL) -> ONNX + INT8 for Raspberry Pi

Converts the trained Stage 2 severity model (Methodology B: EfficientNet-B0 + Variational Bayesian Last Layer) into an **INT8-quantized ONNX** model plus a tiny NumPy head file, for CPU-only deployment on Raspberry Pi 5.

**Before running, attach these inputs (right panel -> Add Input):**
1. **Your Work** -> the saved version of the Stage 2 VBLL notebook (provides `stage2_vbll/checkpoints/stage2_vbll_best.pt`).
2. The same three datasets used for training (IDRiD, ODIR-5K, APTOS).

**What this notebook produces** (in `/kaggle/working/deploy/`):
- `stage2_vbll_int8.onnx` - INT8 backbone; outputs **both** posterior-mean logits (`logits`) and the 1280-d feature vector (`features`)
- `vbll_head.npz` - VBLL posterior parameters (w_mu, w_log_sigma, b_mu, b_log_sigma) for NumPy uncertainty sampling on-device
- `stage2_deploy.json` - grade names, confidence threshold, pass count, normalisation constants

**Why two outputs:** the INT8 backbone runs once per image; the Bayesian uncertainty (30 weight samples) is then computed in NumPy on the 1280-d features - microseconds, no extra backbone passes. That is what makes VBLL the deployable variant.

**Gates that must pass before download:** ONNX checker, FP32-ONNX vs PyTorch allclose, INT8 parity (accuracy and QWK drift < 0.01 vs FP32).

In [ ]:
# Setup
import os, glob, random, math, json, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, cohen_kappa_score

print("Torch:", torch.__version__)

def find_in_inputs(name):
    hits = [p for p in Path("/kaggle/input").rglob(name) if p.is_file()]
    return hits[0] if hits else None

CKPT = find_in_inputs("stage2_vbll_best.pt")
assert CKPT is not None, "stage2_vbll_best.pt not found - attach the Stage 2 VBLL notebook output (Add Input -> Your Work)"
print("Checkpoint:", CKPT)

IDRID_ROOT = Path("/kaggle/input/datasets/lakshmiprathik/idrid-516/IDRiD")
ODIR_ROOT = Path("/kaggle/input/datasets/lakshmiprathik/odir-5k/ODIR-5K")
APTOS_ROOT = Path("/kaggle/input/datasets/mariaherrerot/aptos2019")
APTOS_CSV = APTOS_ROOT / "train_1.csv"
APTOS_IMAGE_DIR = (APTOS_ROOT / "train_images" / "train_images") if (APTOS_ROOT / "train_images" / "train_images").exists() else (APTOS_ROOT / "train_images")

WORK = Path("/kaggle/working/stage2_export")
CACHE = WORK / "cache"
DEPLOY = Path("/kaggle/working/deploy")
for d in (WORK, CACHE, DEPLOY):
    d.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
SEED = 42
ODIR_CAP = 1000
N_QUANT_CAL = 200
PRIOR_SCALE = 1.0
TRAIN_W_SAMPLES = 10
PREDICT_PASSES = 30
LOW_CONF_STD = 0.15
DEVICE = "cpu"

GRADE_NAMES = {0: "No DR", 1: "Mild NPDR", 2: "Moderate NPDR", 3: "Severe NPDR", 4: "Proliferative DR"}
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()
print("IDRiD:", IDRID_ROOT.exists(), "| ODIR:", ODIR_ROOT.exists(), "| APTOS:", APTOS_ROOT.exists())

## Rebuild the dataset index and splits

Identical logic and seed to the Stage 2 VBLL training notebook, so the test partition used for the parity gate is the same one that produced the reported metrics.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def all_images(root):
    return [p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]

def idrid_rows():
    rows = []
    for split in ["train", "validation", "test"]:
        for grade in range(5):
            folder = IDRID_ROOT / split / str(grade)
            if not folder.exists():
                continue
            for image_path in all_images(folder):
                rows.append({"path": str(image_path), "grade": grade, "source": "idrid", "split": split})
    return rows

def aptos_rows():
    aptos_df = pd.read_csv(APTOS_CSV)
    rows = []
    for row in aptos_df.itertuples(index=False):
        image_id = str(row.id_code)
        candidates = [APTOS_IMAGE_DIR / f"{image_id}.png",
                      APTOS_IMAGE_DIR / f"{image_id}.jpg",
                      APTOS_IMAGE_DIR / f"{image_id}.jpeg"]
        image_path = next((p for p in candidates if p.exists()), None)
        if image_path is not None:
            rows.append({"path": str(image_path), "grade": int(row.diagnosis), "source": "aptos", "split": "all"})
    print("APTOS images matched:", len(rows))
    return rows

def odir_rows(cap=ODIR_CAP):
    image_paths = all_images(ODIR_ROOT)
    if len(image_paths) > cap:
        idx = np.random.RandomState(SEED).choice(len(image_paths), size=cap, replace=False)
        image_paths = [image_paths[i] for i in idx]
    return [{"path": str(p), "grade": 0, "source": "odir", "split": "all"} for p in image_paths]

rows = idrid_rows() + aptos_rows() + odir_rows()
df = pd.DataFrame(rows).drop_duplicates(subset="path").reset_index(drop=True)
df["grade"] = df["grade"].astype(int)
print("Total unique images:", len(df))

def crop_square(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > 7
    if mask.sum() > 100:
        ys, xs = np.where(mask)
        img = img[ys.min():ys.max()+1, xs.min():xs.max()+1]
    h, w = img.shape[:2]
    s = max(h, w)
    canvas = np.zeros((s, s, 3), np.uint8)
    y = (s - h)//2; x = (s - w)//2
    canvas[y:y+h, x:x+w] = img
    return canvas

def preprocess(path, size=IMG_SIZE):
    img = cv2.imread(str(path))
    if img is None:
        return np.zeros((size, size, 3), np.uint8)
    img = crop_square(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

def save_cache(frame):
    cached = []
    for i, r in tqdm(frame.iterrows(), total=len(frame), desc="preprocessing"):
        out = CACHE / (str(i) + ".png")
        if not out.exists():
            cv2.imwrite(str(out), preprocess(r.path))
        cached.append(str(out))
    frame = frame.copy()
    frame["cache_path"] = cached
    return frame

df = save_cache(df)

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["grade"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["grade"])
splits = {"train": train_df.reset_index(drop=True),
          "val": val_df.reset_index(drop=True),
          "test": test_df.reset_index(drop=True)}
for k, v in splits.items():
    print(k, len(v), dict(v["grade"].value_counts().sort_index()))

## Load the trained VBLL model and export ONNX (two outputs)

In [ ]:
class VBLLClassifier(nn.Module):
    def __init__(self, in_features, out_features, prior_scale=PRIOR_SCALE, train_samples=TRAIN_W_SAMPLES):
        super().__init__()
        self.prior_scale = prior_scale
        self.train_samples = train_samples
        self.w_mu = nn.Parameter(torch.randn(out_features, in_features) * 0.05)
        self.w_log_sigma = nn.Parameter(torch.full((out_features, in_features), -5.0))
        self.b_mu = nn.Parameter(torch.zeros(out_features))
        self.b_log_sigma = nn.Parameter(torch.full((out_features,), -5.0))
    def forward(self, f):
        return F.linear(f, self.w_mu, self.b_mu)

class VBLLNet(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        in_features = backbone.classifier[1].in_features
        self.vbll = VBLLClassifier(in_features, 5)
    def extract_features(self, x):
        f = self.features(x)
        f = self.avgpool(f)
        return torch.flatten(f, 1)
    def forward(self, x):
        return self.vbll(self.extract_features(x))

model = VBLLNet(models.efficientnet_b0(weights=None))
ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded VBLL model from epoch {ckpt['epoch']} (val QWK={ckpt['qwk']:.4f}, val acc={ckpt['acc']:.4f})")

class ExportWrapper(nn.Module):
    """Returns (posterior-mean logits, 1280-d features) so the device can do
    deterministic grading from `logits` AND Bayesian uncertainty sampling on `features`."""
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, x):
        f = self.net.extract_features(x)
        return self.net.vbll(f), f

wrapper = ExportWrapper(model).eval()

import onnx
import onnxruntime as ort

ONNX_FP32 = WORK / "stage2_vbll_fp32.onnx"
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    wrapper, dummy, str(ONNX_FP32),
    input_names=["input"], output_names=["logits", "features"],
    opset_version=17, dynamo=False,
)
onnx.checker.check_model(str(ONNX_FP32))
print("ONNX exported and valid:", ONNX_FP32, f"({ONNX_FP32.stat().st_size / 1e6:.1f} MB)")

sess = ort.InferenceSession(str(ONNX_FP32), providers=["CPUExecutionProvider"])
x_np = np.random.RandomState(0).randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
with torch.no_grad():
    t_logits, t_feats = wrapper(torch.from_numpy(x_np))
o_logits, o_feats = sess.run(None, {"input": x_np})
for name, t, o in [("logits", t_logits, o_logits), ("features", t_feats, o_feats)]:
    ok = np.allclose(t.numpy(), o, atol=1e-4)
    print(f"  {name}: max abs diff {np.abs(t.numpy() - o).max():.2e} -> {'OK' if ok else 'MISMATCH'}")
    assert ok, f"FP32 ONNX mismatch on {name}"
print("FP32 ONNX matches PyTorch.")

## INT8 static quantization

In [ ]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType

input_name = ort.InferenceSession(str(ONNX_FP32), providers=["CPUExecutionProvider"]).get_inputs()[0].name
print("ONNX input name:", input_name)

def load_batch(paths, batch_size=16):
    imgs = []
    for p in paths:
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        x = (img.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
        imgs.append(x.transpose(2, 0, 1))
        if len(imgs) == batch_size:
            yield np.stack(imgs).astype(np.float32)
            imgs = []
    if imgs:
        yield np.stack(imgs).astype(np.float32)

class FundusCalibrationReader(CalibrationDataReader):
    def __init__(self, paths, in_name, batch_size=16):
        self.gen = load_batch(paths, batch_size)
        self.in_name = in_name
    def get_next(self):
        try:
            return {self.in_name: next(self.gen)}
        except StopIteration:
            return None

cal_paths = splits["train"].sample(min(N_QUANT_CAL, len(splits["train"])), random_state=SEED)["cache_path"].tolist()
ONNX_INT8 = WORK / "stage2_vbll_int8.onnx"
t0 = time.time()
quantize_static(
    model_input=str(ONNX_FP32),
    model_output=str(ONNX_INT8),
    calibration_data_reader=FundusCalibrationReader(cal_paths, input_name),
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
)
print(f"Quantized in {time.time() - t0:.0f}s")
print(f"FP32: {ONNX_FP32.stat().st_size / 1e6:.1f} MB -> INT8: {ONNX_INT8.stat().st_size / 1e6:.1f} MB")
onnx.checker.check_model(str(ONNX_INT8))
print("INT8 ONNX valid.")

## Parity gate: INT8 must behave like FP32

Deterministic posterior-mean predictions over the held-out test split (same partition as training). The gate passes only if both accuracy and QWK drift less than 0.01 between FP32-ONNX and INT8-ONNX.

In [ ]:
sess_f32 = ort.InferenceSession(str(ONNX_FP32), providers=["CPUExecutionProvider"])
sess_i8 = ort.InferenceSession(str(ONNX_INT8), providers=["CPUExecutionProvider"])

def predict_all(sess, frame, batch_size=32, desc="predict"):
    preds = []
    for xb in tqdm(list(load_batch(frame["cache_path"].tolist(), batch_size)), desc=desc):
        logits, _ = sess.run(None, {input_name: xb})
        preds.append(logits.argmax(1))
    return np.concatenate(preds)

y_true = splits["test"]["grade"].values
pred_f32 = predict_all(sess_f32, splits["test"], desc="fp32 test")
pred_i8 = predict_all(sess_i8, splits["test"], desc="int8 test")

acc_f32 = accuracy_score(y_true, pred_f32)
acc_i8 = accuracy_score(y_true, pred_i8)
qwk_f32 = cohen_kappa_score(y_true, pred_f32, weights="quadratic")
qwk_i8 = cohen_kappa_score(y_true, pred_i8, weights="quadratic")
agree = (pred_f32 == pred_i8).mean()

print(f"Accuracy  FP32: {acc_f32:.4f} | INT8: {acc_i8:.4f} | drift: {abs(acc_f32 - acc_i8):.4f}")
print(f"QWK       FP32: {qwk_f32:.4f} | INT8: {qwk_i8:.4f} | drift: {abs(qwk_f32 - qwk_i8):.4f}")
print(f"Prediction agreement FP32 vs INT8: {agree:.4%}")

gates = {
    "accuracy drift < 0.01": abs(acc_f32 - acc_i8) < 0.01,
    "QWK drift < 0.01": abs(qwk_f32 - qwk_i8) < 0.01,
    "prediction agreement >= 99%": agree >= 0.99,
}
for g, ok in gates.items():
    print(("PASS " if ok else "FAIL ") + g)
print()
print("OVERALL:", "PASS - safe to deploy" if all(gates.values()) else "FAIL - do NOT deploy; investigate quantization first")

## Export the VBLL head + deployment artifacts

In [ ]:
# VBLL posterior parameters -> NumPy (device samples weights without PyTorch)
sd = model.vbll.state_dict()
np.savez(
    DEPLOY / "vbll_head.npz",
    w_mu=sd["w_mu"].cpu().numpy().astype(np.float32),
    w_log_sigma=sd["w_log_sigma"].cpu().numpy().astype(np.float32),
    b_mu=sd["b_mu"].cpu().numpy().astype(np.float32),
    b_log_sigma=sd["b_log_sigma"].cpu().numpy().astype(np.float32),
)
print("vbll_head.npz written:",
      {k: tuple(v.shape) for k, v in np.load(DEPLOY / "vbll_head.npz").items()})

# CPU latency (rough proxy only - benchmark.py gives true numbers on the Pi)
def bench(sess, n=30):
    xb = next(load_batch(cal_paths[:1], 1))
    for _ in range(5):
        sess.run(None, {input_name: xb})
    t0 = time.perf_counter()
    for _ in range(n):
        sess.run(None, {input_name: xb})
    return (time.perf_counter() - t0) / n * 1000

print(f"FP32: {bench(sess_f32):.1f} ms/image | INT8: {bench(sess_i8):.1f} ms/image (Kaggle CPU)")

deploy_cfg = {
    "model": "efficientnet_b0 + VBLL head (Methodology B), INT8 ONNX",
    "img_size": IMG_SIZE,
    "input_name": input_name,
    "output_names": ["logits", "features"],
    "preprocessing": "crop + CLAHE + resize 224, then ImageNet normalisation",
    "imagenet_mean": IMAGENET_MEAN.tolist(),
    "imagenet_std": IMAGENET_STD.tolist(),
    "grade_names": GRADE_NAMES,
    "predict_passes": PREDICT_PASSES,
    "low_conf_std": LOW_CONF_STD,
    "uncertainty": "sample W ~ N(w_mu, exp(2*w_log_sigma)) from vbll_head.npz; 30 passes; confidence = (1 - std[top class]) * 100; std > low_conf_std -> Refer for Manual Review",
    "fp32_test_accuracy": float(acc_f32),
    "fp32_test_qwk": float(qwk_f32),
    "int8_test_accuracy": float(acc_i8),
    "int8_test_qwk": float(qwk_i8),
    "source_notebook": "Stage 2 VBLL (stage2_vbll_efficientnetb0)",
}
with open(DEPLOY / "stage2_deploy.json", "w") as f:
    json.dump(deploy_cfg, f, indent=2)
shutil.copy(ONNX_INT8, DEPLOY / "stage2_vbll_int8.onnx")

print()
print("=== DOWNLOAD THESE THREE FILES (right panel -> Output -> deploy/) ===")
print("  deploy/stage2_vbll_int8.onnx")
print("  deploy/vbll_head.npz")
print("  deploy/stage2_deploy.json")
print()
print("Then place them in deploy/raspberry_pi/models/ on your laptop / Raspberry Pi.")